# Review optional district survey context

Read `entities` and `raw_nfhs`. This notebook can run after notebook 02 independently
of the counts and service evidence. NFHS-5 district survey indicators describe an area
in 2019–21. The survey is for context and use in later geographic coverage audits.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Inspect available survey fields

The input is optional outside demo mode. In an authorized Databricks session,
enabling `include_nfhs` in the project config makes notebook 01 load the
configured table. Select the exact indicator names from that input. Parentheses
indicate small samples and asterisks flag suppressed values in the source.

In [ ]:
from facility_tool.profiles import district_context, field_profile

entities = STORE.read("entities")
nfhs = STORE.read("raw_nfhs")
show(field_profile(nfhs), limit=30)
# Edit these names after inspecting available columns above.
indicators = [field for field in ("hh_improved_water_pct", "hh_electricity_pct") if field in nfhs]

## Join on state and district

Use a unique normalized state/district pair. Postal context can fill in an
otherwise unknown district when the available evidence agrees. Conflicting
geographies and duplicate survey keys remain visible for review. Survey
indicators stay at district scope and outside facility-level service evidence.

In [ ]:
if not nfhs.empty and not indicators:
    print("Choose indicator columns above to attach survey values.")
context, ambiguous = district_context(entities, nfhs, indicators)
STORE.write("district_context", context)
STORE.write("ambiguous_nfhs_districts", ambiguous)
if context.empty:
    print("No district survey rows available in this snapshot.")
else:
    show(context["nfhs_join"].value_counts().rename("facilities").reset_index())
    indicator_columns = [column for column in context.columns if any(column.startswith(name) for name in indicators)]
    if indicator_columns:
        show(context[["state_ut", "district_name", *indicator_columns]].drop_duplicates())
show(ambiguous)